# NB-06 — LLM Backbone Integration

**Goal:** Connect CLIP + projector + Qwen2-VL and run your first multimodal QA.

Two paths:
1. **Native** — Qwen2-VL built-in vision (baseline)
2. **Custom** — our CLIP + MLP projector → `inputs_embeds` (learning pipeline)

---

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import time
import torch
import requests
from io import BytesIO
from PIL import Image
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

from src.llm.backbone import MultimodalLLM

DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
DEVICE = "cpu"

Device: mps


In [2]:
def load_image(url: str) -> Image.Image:
  resp = requests.get(url, timeout=15)
  resp.raise_for_status()
  return Image.open(BytesIO(resp.content)).convert("RGB")

IMG_URL = "https://images.unsplash.com/photo-1518717758536-85ae29035b6d?auto=format&fit=crop&w=224&q=80"
image = load_image(IMG_URL)
question = "What animal is in this image? Answer briefly."
image = image.resize((224, 224))

## 1. Load pipeline from config

First run downloads ~4GB (CLIP + Qwen2-VL-2B). Use `encoder_on_cpu=True` if GPU memory is tight.

In [3]:
CONFIG = Path("..") / "config" / "model_config.yaml"

print("Loading MultimodalLLM (this may take a few minutes)...")
model = MultimodalLLM.from_config(
    CONFIG,
    device=DEVICE,
    encoder_on_cpu=(DEVICE == "mps"),  # save MPS memory for LLM
)
print(f"LLM hidden size: {model.llm_hidden_size}")

Loading MultimodalLLM (this may take a few minutes)...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

2026-05-26 16:40:55.683 | INFO     | src.llm.backbone:__init__:114 - Loading LLM: Qwen/Qwen2-VL-2B-Instruct


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

2026-05-26 16:41:09.689 | INFO     | src.llm.backbone:__init__:120 - MultimodalLLM ready | device=cpu | llm_hidden=1536 | encoder_on_cpu=False


LLM hidden size: 1536


## 2. Native baseline (Qwen2-VL vision tower)

In [4]:
t0 = time.perf_counter()
native_answer = model.generate_native(image, question, max_new_tokens=128)
native_latency = time.perf_counter() - t0

print(f"Latency: {native_latency:.2f}s")
print(f"Answer (native): {native_answer}")

2026-05-26 16:41:17.359 | INFO     | src.llm.backbone:generate_native:421 - Native generate | latency=7.61s | new_tokens=2


Latency: 7.61s
Answer (native): dog


## 3. Custom pipeline (CLIP + MLP projector)

In [5]:
inputs = model.prepare_inputs(image, question)
print("Merged input layout: [visual_tokens] + [text_tokens]")
print(f"  visual tokens: {inputs['num_visual_tokens']}")
print(f"  text tokens:   {inputs['num_text_tokens']}")
print(f"  total tokens:  {inputs['total_tokens']}")
print(f"  inputs_embeds: {tuple(inputs['inputs_embeds'].shape)}")

Merged input layout: [visual_tokens] + [text_tokens]
  visual tokens: 256
  text tokens:   10
  total tokens:  266
  inputs_embeds: (1, 266, 1536)


In [6]:
t0 = time.perf_counter()
custom_answer = model.generate(image, question, max_new_tokens=128)
custom_latency = time.perf_counter() - t0

print(f"Latency: {custom_latency:.2f}s")
print(f"Answer (custom): {custom_answer}")

2026-05-26 16:41:21.742 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=256 text_tokens=10 total=266
2026-05-26 16:41:50.260 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=28.87s | ~4.4 tok/s


Latency: 28.87s
Answer (custom): "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image?


## 4. Side-by-side comparison

In [7]:
print("=" * 60)
print(f"Question: {question}")
print("-" * 60)
print(f"Native ({native_latency:.1f}s): {native_answer}")
print(f"Custom ({custom_latency:.1f}s): {custom_answer}")
print("=" * 60)
print("\nNote: Custom path uses untrained CLIP+projector — quality improves after fine-tuning (Phase 5).")

Question: What animal is in this image? Answer briefly.
------------------------------------------------------------
Native (7.6s): dog
Custom (28.9s): "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image? Answer: Aardvark. "What animal is in this image?

Note: Custom path uses untrained CLIP+projector — quality improves after fine-tuning (Phase 5).


## 5. Multi-turn chat

In [8]:
history = [
    {"role": "user", "content": "What is in this image?"},
]
reply1 = model.chat(history, image=image, max_new_tokens=80, use_native=True)
print("Turn 1:", reply1)

history.append({"role": "assistant", "content": reply1})
history.append({"role": "user", "content": "What color is it mainly?"})
reply2 = model.chat(history, image=image, max_new_tokens=80, use_native=True)
print("Turn 2:", reply2)

2026-05-26 16:42:26.224 | INFO     | src.llm.backbone:generate_native:421 - Native generate | latency=35.84s | new_tokens=20


Turn 1: The image shows a chocolate Labrador Retriever dog with its tongue out, possibly licking something.


2026-05-26 16:42:43.581 | INFO     | src.llm.backbone:generate_native:421 - Native generate | latency=17.32s | new_tokens=7


Turn 2: The dog is mainly brown.


## 6. Edge cases

In [9]:
# Text-only (no image)
text_only = model.generate(None, "What is the capital of France?", max_new_tokens=32)
print("Text-only:", text_only)

# Large image (auto-resized by CLIP processor)
large = image.resize((512, 512))
large_inputs = model.prepare_inputs(large, question)
print(f"Large image visual tokens: {large_inputs['num_visual_tokens']}")

2026-05-26 16:42:43.812 | INFO     | src.llm.backbone:generate:330 - Custom generate | visual_tokens=0 text_tokens=7 total=7
2026-05-26 16:42:50.133 | INFO     | src.llm.backbone:generate:353 - Generation done | latency=6.35s | ~3.9 tok/s


Text-only: France? Paris

What is the capital of France? Paris

What is the capital of France? Paris

What is the
Large image visual tokens: 256


## 7. Memory benchmark (optional)

In [10]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    _ = model.generate_native(image, question, max_new_tokens=64)
    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"Peak GPU memory: {peak_mb:.0f} MB")
elif torch.backends.mps.is_available():
    print("MPS memory stats not available — monitor Activity Monitor during runs.")
else:
    print("Running on CPU — expect slower inference.")

MPS memory stats not available — monitor Activity Monitor during runs.


## ✅ Phase 3 Checklist

- [ ] Native Qwen2-VL baseline produces a sensible answer
- [ ] Custom path shows merged `inputs_embeds` shapes
- [ ] Understand visual tokens prepended before text tokens
- [ ] Multi-turn chat works with `model.chat()`

**Next:** Phase 4 — `MultimodalQAPipeline` in `src/pipeline/multimodal_qa.py`